# Лабораторная работа №4
## Прогнозирование риска изменения доли иностранных инвесторов
### Студент: Шевченко Ю.С., ИБМ 3-63Б

### Ячейка 1: Загрузка библиотек и данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('../data/ownership_data.csv')
print('Размер датасета:', df.shape)
df.head()

### Ячейка 2: Расчёт целевой переменной

In [ ]:
# Расчёт дельты по компаниям
df['DeltaForeign'] = df.groupby('Company')['ForeignShare'].diff().abs()
df['DeltaForeign'] = df['DeltaForeign'].fillna(0)

# Целевая переменная: 1 если скачок >= 20 п.п., иначе 0
df['HighOwnershipVolatility'] = (df['DeltaForeign'] >= 20).astype(int)

# Удаляем 2013 год (нет предыдущего значения)
df = df[df['Year'] > 2013].copy()

print('Размер после очистки:', df.shape)
print('\nРаспределение целевой переменной:')
print(df['HighOwnershipVolatility'].value_counts())
print('\nДоля класса 1:', round(df['HighOwnershipVolatility'].mean()*100, 1), '%')

### Ячейка 3: EDA - Визуализация

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Баланс классов
sns.countplot(x='HighOwnershipVolatility', data=df, palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Распределение целевой переменной')
axes[0, 0].set_xlabel('0=стабильно, 1=скачок')

# 2. Распределение ForeignShare
sns.histplot(data=df, x='ForeignShare', kde=True, ax=axes[0, 1], color='skyblue')
axes[0, 1].set_title('Распределение доли иностранных инвесторов')

# 3. DeltaForeign по компаниям
sns.boxplot(data=df, x='Company', y='DeltaForeign', ax=axes[1, 0], palette='pastel')
axes[1, 0].set_title('Дельта доли по компаниям')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Корреляционная матрица
corr = df[['ForeignShare', 'DeltaForeign', 'HighOwnershipVolatility']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1, 1])
axes[1, 1].set_title('Корреляционная матрица')

plt.tight_layout()
plt.show()

### Ячейка 4: Предобработка данных

In [ ]:
# Подготовка признаков и целевой переменной
X = df.drop(columns=['HighOwnershipVolatility', 'Year', 'Company'])
y = df['HighOwnershipVolatility']

# Предобработка: масштабирование числовых + кодирование категориальных
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['ForeignShare', 'DeltaForeign']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['Sector'])
    ]
)

X_proc = preprocessor.fit_transform(X)

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.25, random_state=42, stratify=y
)

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

### Ячейка 5: Обучение моделей

In [ ]:
# Создание и обучение моделей
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)
}

results = {}
print('Обучение моделей...\n')

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }
    print(f"{name}:")
    print(f"  F1-score: {results[name]['F1']:.3f}")
    print(f"  ROC-AUC:  {results[name]['ROC-AUC']:.3f}\n")

### Ячейка 6: Оценка качества моделей

In [ ]:
# Выбор лучшей модели
best_name = max(results, key=lambda x: results[x]['F1'])
print('Лучшая модель:', best_name)

# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, results[best_name]['y_pred'])
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix\n({best_name})')

# ROC Curve
RocCurveDisplay.from_estimator(results[best_name]['model'], X_test, y_test, ax=axes[1])
axes[1].set_title('ROC-кривая')

plt.tight_layout()
plt.show()

# Таблица метрик
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['Accuracy'] for m in results],
    'Precision': [results[m]['Precision'] for m in results],
    'Recall': [results[m]['Recall'] for m in results],
    'F1-score': [results[m]['F1'] for m in results],
    'ROC-AUC': [results[m]['ROC-AUC'] for m in results]
})
print('\nТаблица метрик:')
print(metrics_df.to_string(index=False))

### Ячейка 7: Важность признаков

In [ ]:
# Анализ важности признаков для лучшей модели
feat_names = preprocessor.get_feature_names_out()
importances = results[best_name]['model'].feature_importances_
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
feat_imp.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title(f'Важность признаков\n({best_name})', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print('\nТоп-3 важных признака:')
for i, (feat, imp) in enumerate(feat_imp.head(3).items(), 1):
    print(f"{i}. {feat}: {imp:.3f}")

### Ячейка 8: Бизнес-выводы

## Бизнес-рекомендации:

1. **Мониторинг в периоды нестабильности**: Компании металлургического сектора демонстрируют более высокую волатильность структуры капитала. Рекомендуется усилить мониторинг реестра акционеров в периоды геополитической напряжённости.

2. **Управление рисками**: Резкие изменения доли иностранных инвесторов (≥20 п.п.) требуют превентивных мер - формирования пула якорных российских инвесторов.

3. **IR-стратегия**: Дифференцированный подход к investor relations в зависимости от сектора экономики и текущего уровня иностранного участия.